In [2]:
import os, re
from collections import defaultdict
from tqdm import tqdm

folder = r"\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\xfits"
SETS  = [1, 2, 3,4,5,6,7,8,9,10,11,12]   
n_H   = 43         
# N_FOV = None       

pat = re.compile(r"Conv_zscan__(\d+)--H(\d+)_.*?set(\d+)--(col\d+__Xhfits|dapiFeatures)\.npz$")

files = defaultdict(set)      # (set, H, fov) -> {"xh", "dapi"}
fovs  = defaultdict(set)      # set -> all FOVs seen
with os.scandir(folder) as it:
    for e in tqdm(it, desc="Scanning files", unit="file"):
        m = pat.match(e.name)
        if not m or int(m[3]) not in SETS:
            continue
        fov, H, s = int(m[1]), int(m[2]), int(m[3])
        files[(s, H, fov)].add("dapi" if m[4] == "dapiFeatures" else "xh")
        fovs[s].add(fov)

for s in SETS:
    total = len(fovs[s])
    # an FOV is "done" for an H when both the Xhfits and dapiFeatures files exist
    done = [sum(len(files[(s, H, f)]) == 2 for f in fovs[s]) for H in range(n_H)]
    n_H_done = sum(total > 0 and d == total for d in done)

    print(f"\n=== set{s}: {total} FOVs ===")
    cells = [f"H{H:<2} {d:>4}/{total}{' ✓' if d == total else '  '}" for H, d in enumerate(done)]
    for i in range(0, n_H, 6):
        print("   " + " | ".join(cells[i:i + 6]))
    status = "DONE" if n_H_done == n_H else "NOT done"
    print(f"-> {n_H_done}/{n_H} H complete | set{s} {status}")

Scanning files: 121381file [00:50, 2427.08file/s]


=== set1: 121 FOVs ===
   H0     0/121   | H1   121/121 ✓ | H2   121/121 ✓ | H3   121/121 ✓ | H4   121/121 ✓ | H5   121/121 ✓
   H6   121/121 ✓ | H7   121/121 ✓ | H8   121/121 ✓ | H9   121/121 ✓ | H10  121/121 ✓ | H11  121/121 ✓
   H12  121/121 ✓ | H13  121/121 ✓ | H14  121/121 ✓ | H15  121/121 ✓ | H16  121/121 ✓ | H17  121/121 ✓
   H18  121/121 ✓ | H19  121/121 ✓ | H20  121/121 ✓ | H21  121/121 ✓ | H22  121/121 ✓ | H23  121/121 ✓
   H24  121/121 ✓ | H25  121/121 ✓ | H26  121/121 ✓ | H27  121/121 ✓ | H28  121/121 ✓ | H29  121/121 ✓
   H30  121/121 ✓ | H31  121/121 ✓ | H32  121/121 ✓ | H33  121/121 ✓ | H34  121/121 ✓ | H35  121/121 ✓
   H36  121/121 ✓ | H37  121/121 ✓ | H38  121/121 ✓ | H39  121/121 ✓ | H40  121/121 ✓ | H41  121/121 ✓
   H42  121/121 ✓
-> 42/43 H complete | set1 NOT done

=== set2: 130 FOVs ===
   H0     0/130   | H1   130/130 ✓ | H2   130/130 ✓ | H3   130/130 ✓ | H4   130/130 ✓ | H5   130/130 ✓
   H6   130/130 ✓ | H7   130/130 ✓ | H8   130/130 ✓ | H9   130/130 ✓ | H10

In [12]:
def ranges(nums):
    out, start = [], None
    for i, n in enumerate(nums):
        if start is None: start = n
        if i == len(nums) - 1 or nums[i + 1] != n + 1:
            out.append(f"{start}-{n}" if start != n else f"{start}")
            start = None
    return ", ".join(out)

for s in SETS:
    for H in range(n_H):
        missing = [f for f in sorted(fovs[s]) if len(files[(s, H, f)]) < 2]
        if missing:
            print(f"set{s} H{H}: {len(missing)} missing -> FOV {ranges(missing)}")

set1 H21: 16 missing -> FOV 7, 9, 14, 60, 80, 83, 86, 100, 106-108, 111, 115-116, 118-119
set1 H22: 16 missing -> FOV 7, 9, 14, 60, 80, 83, 86, 100, 106-108, 111, 115-116, 118-119
set1 H23: 16 missing -> FOV 7, 9, 14, 60, 80, 83, 86, 100, 106-108, 111, 115-116, 118-119
set1 H24: 16 missing -> FOV 7, 9, 14, 60, 80, 83, 86, 100, 106-108, 111, 115-116, 118-119
set1 H25: 16 missing -> FOV 7, 9, 14, 60, 80, 83, 86, 100, 106-108, 111, 115-116, 118-119
set1 H26: 16 missing -> FOV 7, 9, 14, 60, 80, 83, 86, 100, 106-108, 111, 115-116, 118-119
set1 H27: 16 missing -> FOV 7, 9, 14, 60, 80, 83, 86, 100, 106-108, 111, 115-116, 118-119
set1 H28: 16 missing -> FOV 7, 9, 14, 60, 80, 83, 86, 100, 106-108, 111, 115-116, 118-119
set1 H29: 16 missing -> FOV 7, 9, 14, 60, 80, 83, 86, 100, 106-108, 111, 115-116, 118-119
set1 H30: 16 missing -> FOV 7, 9, 14, 60, 80, 83, 86, 100, 106-108, 111, 115-116, 118-119
set1 H31: 16 missing -> FOV 7, 9, 14, 60, 80, 83, 86, 100, 106-108, 111, 115-116, 118-119
set1 H32: 